ST554_Final Project   
Author: Joy Zhou  
Date: 4/21/2026

# Introduction   
This project is designed to assess the ability to use Apache Spark for processing streaming data and for training machine learning models in a scalable computing environment.


We will use a modified dataset from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city) on power consumption in Tetouan City, the project focuses on modeling the relationship between electricity consumption across different city zones and key influencing factors such as time of day, temperature, and humidity.   


By leveraging Spark's data processing and machine learning libraries, a predictive model will be developed using historical data and then applied to incoming data streams to perform real-time monitoring and prediction of power consumption. This project integrates concepts from big data processing, machine learning, and streaming analytics, highlighting the practical application of Spark in handling real-world, data-intensive problems.


# Part 1 Fitting the Model



The dataset power_ml_data.csv is available at https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv.
The data should first be loaded into a pandas DataFrame using the `pd.read_csv()` function and then converted into a Spark DataFrame. In this analysis, `Power_Zone_3` is treated as the response variable, while all remaining variables are used as predictors.

## Read in data

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 22:30:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/28 22:30:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/28 22:30:15 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/28 22:30:15 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


- First we read `power_ml_data` into a standard pandas data frame named `power_data` using the pd.read_csv() function.


In [2]:
power_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


- Convert `power_data` to a spark data frame `power`

In [3]:
#Convert this to a spark data frame
power = spark.createDataFrame(power_data)
power.show(5)
#We are going to treat the Power_Zone_3 variable as our response variable

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## Data transformation
We will fit an elastic net regression using cross-validation (CV), without performing an explicit training-test split. Instead, model selection and performance assessment are conducted entirely through cross-validation on the available dataset.   

Feature transformations are performed using Spark MLlib utilities, and the transformed variables are assembled into a machine learning pipeline to ensure a consistent and reproducible workflow.

- let's check the varaible types by inspectiong the dataset schema.


In [4]:
power.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



### Hour variable binary transformation
- Inspection of the schema shows that the Hour column is defined as a LongType. We therefore apply an SQLTransformer to cast this variable to DoubleType.

In [5]:
from pyspark.ml.feature import SQLTransformer
sqlTrans = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__")

To inspect the results of the SQL transformation, we apply sqlTrans to the power dataset and displayed a sample of the transformed records.

In [6]:
#inspect of records of sqlTrans
transformed_power = sqlTrans.transform(power)
transformed_power.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335

- We apply a binarization step to the Hour variable using a cutoff of 6.5, creating a new binary feature, night_vs_day, to indicate nighttime (Hour < 6.5) versus daytime (Hour ≥ 6.5) for downstream modeling.
    

In [7]:
from pyspark.ml.feature import Binarizer

binaryTrans = Binarizer(threshold=6.5, inputCol="Hour_double", outputCol="night_vs_day")

#inspect the transformer results
binary = binaryTrans.transform(
    sqlTrans.transform(power))
binary.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0

### One‑Hot encoding for the Month variable
Although the Month variable is stored as a LongType, it represents a categorical feature rather than a continuous numeric value. Therefore, we apply one-hot encoding to the `Month` column to generate binary indicator variables, allowing the model to capture seasonal effects without assuming an ordinal or linear relationship among months.

In [8]:
from pyspark.ml.feature import OneHotEncoder, VectorAssembler

# Location one-hot encoding
month_encoder = OneHotEncoder(inputCol="Month", outputCol="Month_ind", dropLast=True)

- We fit the month_encoder model to inspect the transformed records to ensure that the one-hot encoding is performed correctly.

In [9]:
#inspect the one-hot encode results
model = month_encoder.fit(binary)
encoded = model.transform(binary)
encoded.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|     Month_ind|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|       

### PAC on environmental variables 
- Principal Component Analysis (PCA) is used to reduce the dimensionality of the environmental variables Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows. The variables are assembled into a feature vector and standardized using [StandardScaler](https://www.sparkcodehub.com/pyspark/mllib/standard-scaler) to mitigate the influence of variables with larger magnitudes. PCA is then applied to extract two uncorrelated principal components, which provide lower‑dimensional representations for subsequent analysis and modeling.


In [10]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml import Pipeline

#assemble raw features
assembler = VectorAssembler(inputCols=["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol="pcaFeatures")

# Scale features (mean=0, std=1)
scaler = StandardScaler(inputCol="pcaFeatures", outputCol="scaledFeatures", withMean=True, withStd=True)

#PCA on scaled features
pca = PCA(k=2, inputCol="scaledFeatures", outputCol="pcaOutput")

#inspect the results
#set up pipeline
pipeline = Pipeline(stages=[assembler, scaler, pca])

# Fit and transform
pca_model = pipeline.fit(encoded)
pca_transformed = pca_model.transform(encoded)

pca_transformed.select("scaledFeatures", "pcaOutput").show(5, truncate=False)

+----------------------------------------------------------------------------------------------------+---------------------------------------+
|scaledFeatures                                                                                      |pcaOutput                              |
+----------------------------------------------------------------------------------------------------+---------------------------------------+
|[-2.1079477438809433,0.3542085264292604,-0.7996341497278044,-0.6900839531060153,-0.6025312519793238]|[2.069074321346372,0.8078678829472264] |
|[-2.1328903699941857,0.3991947174962608,-0.7996341497278044,-0.6900121009460847,-0.6028048802947571]|[2.102921063880654,0.8152476222806395] |
|[-2.1502641992178924,0.3991947174962608,-0.800911098051248,-0.690042354487108,-0.6026841619203012]  |[2.1120662633791016,0.8227151924829962]|
|[-2.183291676554047,0.4313277111155467,-0.7996341497278044,-0.6899326854008982,-0.6027163534868227] |[2.14350318474222,0.8329135817940969]  |

As the output above, we combined several environmental measurements into a single dataset and used PCA to compress them into two summary variables that capture most of the information. These summaries were then used instead of the original variables in later analyses.

### Create label column
- we will rename our response variable `Power_Zone_3` as label

In [11]:
sqlLabel = SQLTransformer(
    statement = """
                SELECT *, Power_Zone_3 AS label FROM __THIS__
                """
)

- Use `VectorAssembler()` to put all predictors into a single features vector, which includes:   
    - two fitted PCA features (pcaOutput)      
    - a binary Hour variable (night_vs_day)   
    - Power_Zone_1   
    - Power_Zone_2   
    - Month indicator variables (Month_ind)   
Since the PCA features were standardized in previous steps, the additional non‑PCA features will also be standardized to ensure consistency across all predictors.

In [12]:
#assemble assembleFeatures
features_assembler = VectorAssembler(
    inputCols=["pcaOutput", "night_vs_day", "Power_Zone_1", "Power_Zone_2", "Month_ind"],
    outputCol="assembledFeatures"
)

#scale again after adding non‑PCA features.
final_scaler = StandardScaler(
    inputCol="assembledFeatures",
    outputCol="features",
    withMean=True,
    withStd=True
)

assembled_df = features_assembler.transform(pca_transformed)
scaler_model = final_scaler.fit(assembled_df)
featurestrans = scaler_model.transform(assembled_df)

# Inspect assembled features
featurestrans.select("features").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                                                                                                                                                                                                              |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[1.355834830

From the output above, we confirmed that the transformation pipeline was completed and the the dataset is ready for modeling.

## Fit an Elastic Net Model
We next fit an Elastic Net regression model using the `CrossValidator()` and `LinearRegression()` function. Model hyperparameters are selected via cross-validation over a predefined grid consisting of all combinations of the following values:   

- regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

We are now ready to set up the pipeline, which includes transcormations and the model to be fitted. We use the `Pipeline()` function from the `pyspark.ml` module to set up a squential workflow of transformations/estimators.

In [13]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline

# Elastic Net regression
lr = LinearRegression(featuresCol='features', labelCol='label') #create a liner regression instance

#define parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

- Set up the pipeline 

In [14]:
pipeline = Pipeline(stages=[
    sqlTrans,              # Cast hour 
    binaryTrans,           # night_vs_day
    month_encoder,         # Month one-hot encoding
    sqlLabel,              # Rename response variable
    assembler,             # Assemble environmental vars to create pcaFeatures
    scaler,                # Scale pcaFeatures
    pca,                   # PCA produces pcaOutput
    features_assembler,    # Combine pcaOutput + indicators to get assembledFeatures
    final_scaler,          # Scale final feature vector to features
    lr                     # Elastic Net regression model
])

We apply `CrossValidator()` to carry out 5-fold cross-validation over the specified hyperparameter grid. Model performance is evaluated using `RegressionEvaluator()`, with RMSE specified via the metricName argument. This procedure will split the dataset into five folds. In each iteration, the model is trained on four folds and validated on the remaining fold for every hyperparameter combination. The performance metric (rmse) is then averaged across all five folds, and the model with the lowest average rmse is selected as the best model.

In [15]:
from pyspark.ml.evaluation import RegressionEvaluator
#initialize cross-validator
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    ),
    numFolds=5,        #5-fold cross-validation
    parallelism=32,
    seed = 42
)
#fitting the model, and choose the best set of parameters
cv_model = cv.fit(power)

Evaluating the best model
- We will use the bestModel attribute to retrive the best model from cross validation
- Then, extract the final stage of the pipeline, the trained LinearRegression model, from the selected best model for further evaluation and interpretation.

In [16]:
# Retrieves the best pipeline model from the cross-validation
best_model = cv_model.bestModel
#Extracts the last stage of the pipeline
best_lr = best_model.stages[-1]  # LinearRegression is last stage

Report the optimal values of the tuning parameters (regParam and elasticNetParam) selected by cross‑validation using `getRegParam()` and `getElasticNetParam()` methods.

In [17]:
# Printing parameters for verification
print(f"Best regParam: {best_lr.getRegParam()}")
print(f"Best elasticNetParam: {best_lr.getElasticNetParam()}")

Best regParam: 0.05
Best elasticNetParam: 0.05


Both parameters were selected as 0.05 because cross‑validation identified a model with mild, Ridge‑dominant regularization that best balances bias and variance for the standardized feature set.

Report the CV error   
- The cross‑validation error is computed as the average RMSE across the five folds, evaluated over all hyperparameter combinations.

In [18]:
#report CV error
avg_rmse = min(cv_model.avgMetrics)
print(f"Best CV RMSE: {avg_rmse:.4f}")

Best CV RMSE: 2174.9962


Report training set RMSE
- The fitted regression model with the best parameters now has a transform method that can be used to make predictions using the entire dataset. The training set RMSE is obtained by applying the best fitted pipeline model to the entire dataset using the transform() method and evaluating prediction error using RMSE.

In [19]:
#apply the best fitted pipeline model to the entire dataset
train_predictions = best_model.transform(power)

#define evaluator
evaluator=RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    )
# Evaluate RMSE
train_rmse = evaluator.evaluate(train_predictions)
print(f"Training RMSE: {train_rmse:.4f}")

Training RMSE: 2174.4490


The similarity between the cross‑validation RMSE (2174.9962) and the training RMSE (2174.4490) demonstrates that the model achieves stable generalization with minimal overfitting.

The model outputs are used to generate predictions, from which a residual column is computed as the difference between the observed label and the predicted value. The `.withColumn()` method is used to create this residual. A resulting data frame is then displayed containing the residuals, the label column, and the model predictions.

In [24]:
from pyspark.sql.functions import col

residuals_df = train_predictions.withColumn("residual", col("label") - col("prediction"))

residuals_df.select( "label", "prediction", "residual",).show(10, truncate=False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20936.22684100606 |-695.2629810060607|
|20131.08434|18701.58646809512 |1429.4978719048813|
|19668.43373|18237.189469773388|1431.2442602266128|
|18899.27711|17615.892216250577|1283.3848937494222|
|18442.40964|17017.38184268389 |1425.02779731611  |
|18130.12048|16540.614302205624|1589.5061777943774|
|17945.06024|16114.684233894424|1830.3760061055746|
|17459.27711|15742.03833420217 |1717.2387757978286|
|17025.54217|15282.557973009165|1742.9841969908357|
|16794.21687|14934.899321422641|1859.317548577359 |
+-----------+------------------+------------------+
only showing top 10 rows


## Conclusion
Using the Power dataset, an elastic net linear regression model was fitted through a pipeline and tuned via 5‑fold cross‑validation, with RMSE serving as the evaluation metric. The optimal regularization parameters were selected based on cross‑validated performance. The best‑fitted pipeline model was then applied to the entire dataset to compute the training set RMSE, which was 2174.449. Prediction residuals were subsequently calculated as the difference between the observed and predicted values.
During this process, transformations were applied to the hour and month variables, and all features were standardized prior to PCA and model fitting to mitigate the influence of variables with larger magnitudes. The use of a pipeline enabled efficient tracking of feature transformations and streamlined the modeling workflow, while Spark facilitated scalable and efficient model fitting. In the next section, methods for handling streaming data will be explored.

# Streaming Part


## Reading a stream
We set up a Structured Streaming read operation that monitors a specified folder for incoming CSV files, which created by .py file).

In [39]:
#create spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

- The schema from the power DataFrame is extracted and reused to ensure consistent column types when setting up the streaming CSV source. The header option is set to True so that all incoming files are correctly read with column headers.

In [40]:
from pyspark.sql.functions import current_timestamp

power_df = power.withColumn("timestamp", current_timestamp())
power_schema = power_df.schema

stream_df = (
    spark.readStream
        .schema(power_schema)
        .option("header", True)
        .csv("csv_files")
)

## Transform/Aggregation Step
The streaming dataset is processed using two separate transformations derived from the same input stream, following these steps:

- The first transformation applies the fitted pipeline model to generate prediction values, residuals are computed as the difference between the observed values and the model predictions.

In [41]:
from pyspark.sql.functions import col

predictions_df = (
    best_model
        .transform(stream_df) 
        .withColumn("residual", col("label") - col("prediction"))
        .select("label", "prediction", "residual", "timestamp") 
)

- We will create the second transformation on the original stream. We take the rresponse variable `Power_Zone_3` and rename it to `label`.

In [42]:
#2nd treansformation
label_df = stream_df.withColumnRenamed("Power_Zone_3", "label")

- The two transformed streams are then joined on the common label column using `join()` method.

In [43]:
joined_stream = predictions_df.join(label_df, on="label")

## Write the stream

The `.writeStream` method is used to output the transformed streaming data to the console in append mode. The `.start()` method initiates the streaming query and begins monitoring the input directory for new data. The five CSV files are added to the empty `csv_files` folder one at a time to be processed incrementally.

When We start the query, We read in streaming datasets using the method that created in `stream.py` file. We will submit `stream.py` in a python console.


In [46]:
#Write the transformed streaming data to the console and start the streaming query
query = (
    joined_stream
        .writeStream
        .trigger(processingTime = "1 second") # add a trigger
        .format("console") 
        .outputMode("append")
        .start()
)

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|      label|        prediction|           residual|          timestamp|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|          timestamp|
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|16434.65587| 17960.19637263891|-1525.5405026389126|2026-04-28 23:02:07|       23.4|   51.02|     0.072|                598.6|        282.7|  35466.4918| 22922.60062|    5|  11|2026-04-28 23:02:07|
|13983.61446| 13642.03376844659| 341.58069155341036|2026-04-28 23:02:07|      19.74|    76.2|     0.069|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|      label|        prediction|           residual|          timestamp|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|          timestamp|
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|14315.37994|11185.484306970258| 3129.8956330297424|2026-04-28 23:02:27|      19.87|    91.3|     0.066|                 0.04|        0.107| 30558.24945| 19273.44398|   10|  23|2026-04-28 23:02:27|
|19444.36364| 19199.85761750036| 244.50602249963777|2026-04-28 23:02:27|      19.97|   60.13|     0.086|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|      label|        prediction|           residual|          timestamp|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|          timestamp|
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
| 7610.60241|  9269.23681835775|-1658.6344083577487|2026-04-28 23:02:47|      15.97|   55.84|     0.085|                0.029|        0.126| 21790.76923| 17181.81818|   11|   6|2026-04-28 23:02:47|
|9632.903226| 9019.461622355508|  613.4416036444927|2026-04-28 23:02:47|      11.24|   60.71|     4.921|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|      label|        prediction|           residual|          timestamp|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|          timestamp|
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|11548.91566| 12173.47709174253| -624.5614317425297|2026-04-28 23:03:07|      6.958|    87.0|     0.089|                0.106|          0.1| 23125.06329| 14972.64438|    1|   7|2026-04-28 23:03:07|
|27659.81538|28460.301988409694| -800.4866084096939|2026-04-28 23:03:07|      20.82|    80.1|     0.066|       

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|      label|        prediction|           residual|          timestamp|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|          timestamp|
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|16438.06452|15937.417101617353|  500.6474183826467|2026-04-28 23:03:27|      15.77|   58.64|      4.92|                412.1|        429.0| 32102.80851| 18193.90244|    3|  17|2026-04-28 23:03:27|
|15138.38611|  17372.7286966619|   -2234.3425866619|2026-04-28 23:03:27|      23.02|   66.19|     4.921|       

...

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|      label|        prediction|           residual|          timestamp|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|          timestamp|
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|11759.27052| 10543.66278816908| 1215.6077318309199|2026-04-28 23:06:27|      20.91|   60.12|     4.922|                0.102|        0.078| 26461.96937|  15318.6722|   10|   2|2026-04-28 23:06:27|
|14765.80645|17251.592286555966| -2485.785836555966|2026-04-28 23:06:27|      13.52|    70.7|     0.087|      

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|      label|        prediction|           residual|          timestamp|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|          timestamp|
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|13889.03226|13162.264939030601|  726.7673209693985|2026-04-28 23:06:47|      10.77|    84.8|     0.088|                0.099|        0.134| 22758.12766| 13481.70732|    3|   2|2026-04-28 23:06:47|
|15456.09806|13818.920086419443| 1637.1779735805576|2026-04-28 23:06:47|      25.21|   46.68|     0.203|      

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|      label|        prediction|          residual|          timestamp|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|          timestamp|
+-----------+------------------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|25309.09091|23849.659906563553|1459.4310034364462|2026-04-28 23:07:07|      27.28|   51.65|     4.903|                581.6|         97.7| 36701.62042| 26336.64203|    8|  10|2026-04-28 23:07:07|
|8361.104442| 9808.103632815599|-1446.999190815599|2026-04-28 23:07:07|      12.25|    78.1|      0.08|           

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|      label|        prediction|           residual|          timestamp|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|          timestamp|
+-----------+------------------+-------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|17881.44578|17490.170066735096| 391.27571326490215|2026-04-28 23:07:27|      12.89|   57.13|     0.085|                398.7|        432.0| 33818.73418| 21884.49848|    1|  14|2026-04-28 23:07:27|
|25347.46988| 24281.91941436925| 1065.5504656307494|2026-04-28 23:07:27|      11.46|   53.89|     0.086|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
|      label|        prediction|          residual|          timestamp|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|          timestamp|
+-----------+------------------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-------------------+
| 18599.8794|19144.246421198623|-544.3670211986209|2026-04-28 23:07:47|      13.98|   53.37|     0.085|                113.9|        130.3| 35090.84746| 20972.64438|    2|  18|2026-04-28 23:07:47|
|10095.55822|11536.351908286999|-1440.793688286998|2026-04-28 23:07:47|      17.27|   42.22|     0.087|           

In [ ]:
# Stop the streaming query after processing all input files
query.stop()